# Entropy, Cross-Entropy and KL Divergence
## Information theory for classification — a physicist's tour

This notebook derives the core quantities of information theory from first principles
(Shannon 1948, building on Boltzmann/Gibbs statistical mechanics), proves their key
properties, connects them to thermodynamics and the maximum-entropy principle, and then
shows precisely why **cross-entropy is the loss of choice in classification**.

**TL;DR**
- Information of an event must be additive for independent events → the only continuous
  choice is $I(p) = -\log p$.
- Entropy $H(p) = -\sum_k p_k \log p_k$ is the expected optimal code length — the
  compression limit (Shannon's source coding theorem).
- Cross-entropy $H(p,q) = -\sum_k p_k \log q_k = H(p) + D_{KL}(p\|q)$ is the expected
  length when you code for the wrong distribution $q$; the excess is the KL divergence.
- In classification with one-hot labels, CE collapses to $-\log q[c]$ = negative
  log-likelihood, so minimizing CE **is** maximum likelihood; with softmax logits it has
  the stable form $\ell = \operatorname{logsumexp}(l) - l_c$ and gradient $\hat y - y$.

## 1. Where the idea comes from: physics → communication

**Boltzmann (1870s).** For a macrostate with $W$ microstates, entropy is

$$
S = k_B \ln W
$$

— a measure of how many microscopic configurations realize the same macroscopic state.

**Gibbs (1902).** When microstate $i$ has probability $p_i$, the general form is

$$
S = -k_B \sum_i p_i \ln p_i
$$

which reduces to Boltzmann's $k_B \ln W$ in the microcanonical case ($p_i = 1/W$ on $W$
states, 0 elsewhere). Entropy = (missing information) × $k_B$.

**Shannon (1948), *A Mathematical Theory of Communication*.** Shannon needed a measure of
uncertainty/information in a message source. He dropped the $k_B$, defined

$$
H(p) = -\sum_k p_k \log_2 p_k \quad \text{(bits)}
$$

and proved it is the fundamental limit of compression. Same mathematics as Gibbs entropy —
this is why ML people say "entropy" and physicists say "entropy" and mean the same object.

**Kullback & Leibler (1951)** later isolated the asymmetric distance between two
distributions, $D_{KL}(p\|q)$, which is the "extra cost of using $q$ when truth is $p$.

## 2. Why is information $-\log p$?

Shannon imposed three axioms on the information $I(p)$ carried by an event of probability $p$:

1. **Monotonicity:** rarer events carry more information → $I$ strictly decreasing in $p$
2. **Continuity:** $I(p)$ varies smoothly with $p$
3. **Additivity:** for independent events $x, y$: $I(p_x p_y) = I(p_x) + I(p_y)$

Axiom 3 is a functional equation: $f(xy) = f(x) + f(y)$ for all $x,y \in (0,1]$.
With continuity, the **only** solutions are $f(x) = c\log x$ (Cauchy's equation argument:
additivity forces the log; continuity rules out pathological solutions). Monotonicity
forces $c < 0$. Hence

$$
I(p) = -c \log p
$$

The constant $c$ only picks units: $\log_2$ → **bits**, $\ln$ → **nats**, and in physics
$k_B \ln$ → **joules per kelvin**.

**Examples.** A fair coin: $I = 1$ bit. A fair die: $\log_2 6 \approx 2.58$ bits. Landing
"heads" with $p = 0.9$: $-\log_2 0.9 \approx 0.15$ bits — unsurprising, little information.
The intuition: information = how many yes/no questions the outcome answers.

## 3. Entropy: definition and properties

$$
H(p) = \mathbb{E}_p[I] = -\sum_{k=1}^{K} p_k \log p_k
$$

(with $0\log 0 := 0$). Properties, with proofs:

**(a) Non-negativity.** Each term $-p_k \log p_k \ge 0$ since $p_k \in [0,1]$. So $H \ge 0$,
with $H = 0$ iff one $p_k = 1$ (deterministic — no uncertainty).

**(b) Maximum at the uniform distribution** ($H \le \log K$, equality iff $p_k = 1/K$).
By concavity of $\log$ (Jensen):

$$
H(p) = \sum_k p_k \log\frac{1}{p_k} \;\le\; \log \sum_k p_k \cdot \frac{1}{p_k} = \log K
$$

Uniform = maximum uncertainty = hardest to predict.

**(c) Concavity.** $H$ is concave in $p$: mixing distributions increases entropy,
$H(\lambda p + (1-\lambda) q) \ge \lambda H(p) + (1-\lambda) H(q)$.

**(d) Chain rule.** For a joint distribution:

$$
H(X,Y) = H(X) + H(Y \mid X), \qquad H(Y\mid X) = \sum_x p(x)\, H(p(\cdot \mid x))
$$

**(e) Conditioning reduces entropy:** $H(Y \mid X) \le H(Y)$ — knowing $X$ never increases
your uncertainty about $Y$ on average. Equality iff $X, Y$ independent. (Proof: the gap
is the mutual information, §6.)

**Example.** Coin with $P(H) = q$: $H = -q\log_2 q - (1-q)\log_2(1-q)$ (binary entropy
$h_2(q)$): $h_2(0.5) = 1$ bit max; $h_2(0.1) \approx 0.47$ bits.

## 4. Source coding: why entropy is the compression limit

A **prefix-free code** assigns bit strings to symbols such that no codeword is a prefix of
another (instantaneously decodable).

**Kraft inequality.** Lengths $l_1, \dots, l_K$ are realizable as a prefix code iff

$$
\sum_{k=1}^{K} 2^{-l_k} \le 1
$$

(Sketch: think of the binary tree; a codeword of length $l$ occupies a fraction $2^{-l}$
of the leaves.)

**Shannon's source coding theorem (1948).** For i.i.d. symbols from $p$:

- any uniquely decodable code has expected length $L = \sum_k p_k l_k \ge H(p)$
- there exists a prefix code (e.g. Huffman) with $H(p) \le L < H(p) + 1$

Choosing $l_k \approx -\log_2 p_k$ achieves the bound: frequent symbols get short codes.
Hence

$$
L_{\min} = H(p)
$$

— **entropy is the average number of bits per symbol you cannot beat**, and Huffman codes
come within 1 bit. This is why $H$ deserves the name "the information content of the source":
it is exactly the irreducible size of the data.

## 5. Cross-entropy: the cost of the wrong code

Suppose symbols really come from $p$, but you built your code for a different distribution
$q$, i.e. you assign length $-\log_2 q_k$ to symbol $k$. The expected length is

$$
H(p, q) = \sum_k p_k \, (-\log_2 q_k) = -\sum_k p_k \log q_k \quad \text{(cross-entropy, bits)}
$$

Decompose it:

$$
H(p,q) = -\sum_k p_k \log q_k
       = -\sum_k p_k \log p_k + \sum_k p_k \log\frac{p_k}{q_k}
       = H(p) + D_{KL}(p \,\|\, q)
$$

with the **KL divergence**

$$
D_{KL}(p\,\|\,q) = \sum_k p_k \log\frac{p_k}{q_k} \;\ge\; 0
$$

**Gibbs' inequality — proof.** First, $\ln x \le x - 1$ for all $x > 0$ (the graph of $\ln$
lies below its tangent at $x=1$; equality iff $x=1$). Then, with $x = q_k/p_k$:

$$
-D_{KL}(p\|q) = \sum_k p_k \ln\frac{q_k}{p_k}
             \le \sum_k p_k\left(\frac{q_k}{p_k} - 1\right)
             = \sum_k q_k - \sum_k p_k = 1 - 1 = 0
$$

So $D_{KL} \ge 0$ with equality iff $p = q$ almost everywhere. The true distribution's own
code is optimal; every other code pays $D_{KL}$ extra bits per symbol on average.

## 6. KL divergence, mutual information, and how they fit together

**Definition.** $D_{KL}(p\|q) = \sum_k p_k \log \frac{p_k}{q_k}$ (nats if $\ln$). In physics
this is called the **relative entropy** $S(p\|q)$.

**Properties.**
- $D_{KL}(p\|q) \ge 0$, equality iff $p = q$ (Gibbs, §5)
- **Asymmetric:** generally $D_{KL}(p\|q) \ne D_{KL}(q\|p)$ — e.g. $p = (0.5, 0.5)$,
  $q = (0.9, 0.1)$: $D_{KL}(p\|q) \approx 0.51$ nats, $D_{KL}(q\|p) \approx 0.37$ nats
- **Convex** in the pair $(p, q)$ — important for optimization
- **Pinsker's inequality:** $D_{KL}(p\|q) \ge \tfrac{1}{2}\lVert p - q\rVert_1^2$ (in nats) — KL
  controls total variation distance
- Not a metric (no triangle inequality, asymmetric) — but a divergence

**Mutual information** ties KL to entropy of joint vs. product:

$$
I(X;Y) = \sum_{x,y} p(x,y) \log\frac{p(x,y)}{p(x)p(y)} = D_{KL}\big(p(x,y)\,\big\|\,p(x)p(y)\big)
$$

$$
I(X;Y) = H(X) - H(X \mid Y) = H(Y) - H(Y \mid X) = H(X) + H(Y) - H(X,Y) \ge 0
$$

Mutual information = how much knowing one variable reduces uncertainty about the other;
it is zero iff independent. And the chain rule for KL:
$D_{KL}(p(x,y)\|q(x,y)) = D_{KL}(p(x)\|q(x)) + \mathbb{E}_{p(x)}[D_{KL}(p(y|x)\|q(y|x))]$.

## 7. The physics connection: free energy and the maximum-entropy principle

**Relative entropy = free-energy difference.** Let $q$ be the thermal (canonical)
distribution at temperature $T$: $q_i = e^{-\beta E_i}/Z$, $\beta = 1/(k_B T)$,
$Z = \sum_i e^{-\beta E_i}$. For any distribution $p$ over the same states:

$$
D_{KL}(p\,\|\,q) = \frac{F[p] - F_{\text{eq}}}{k_B T}, \qquad
F[p] = \langle E\rangle_p - T S[p]
$$

(Verify by expanding both sides: $D_{KL} = -S[p]/k_B + \beta\langle E\rangle_p + \ln Z$.)
So minimizing KL to equilibrium is literally minimizing free energy — the same variational
principle as thermodynamics.

**Maximum-entropy principle (Jaynes).** Given only constraints $\sum_i p_i f_j(i) = F_j$
(e.g. fixed mean energy), the least-biased distribution maximizes $H(p)$. Lagrange
multipliers: maximize $-\sum_i p_i \ln p_i$ s.t. $\sum p_i = 1$, $\sum p_i f_j(i) = F_j$:

$$
\frac{\partial}{\partial p_i}\Big[-\sum_l p_l \ln p_l - \lambda_0(\textstyle\sum_l p_l - 1) - \textstyle\sum_j \lambda_j (\sum_l p_l f_j(l) - F_j)\Big] = 0
$$

$$
\Rightarrow\; -(\ln p_i + 1) - \lambda_0 - \sum_j \lambda_j f_j(i) = 0
\;\Rightarrow\; p_i = \frac{1}{Z}\exp\!\Big(-\sum_j \lambda_j f_j(i)\Big), \quad Z = \sum_i e^{-\sum_j \lambda_j f_j(i)}
$$

The max-entropy distribution is always an **exponential family** (Gibbs/Boltzmann when the
constraint is mean energy: $p_i \propto e^{-\beta E_i}$). No constraints → uniform.
This is the principled answer to "which distribution should I assume?": the one that
encodes nothing beyond what you know.

## 8. From max-entropy to softmax

Apply max-entropy with a constraint on the expected value of a feature vector. For
classification, take the "features" to be the one-hot class indicators with fixed expected
class probabilities. The result is exactly the **softmax**:

$$
p(c \mid x) = \frac{e^{l_c}}{\sum_k e^{l_k}} = \operatorname{softmax}(l)_c
$$

where $l_c$ are the logits. So a softmax classifier is the maximum-entropy model whose
average predicted class scores match the chosen ones — the same exponential-family form
as the Boltzmann distribution. This is why softmax + cross-entropy is the canonical pair:
softmax is max-ent, cross-entropy is its matching loss (§9).

## 9. Why cross-entropy is the classification loss

**Setup.** Model outputs logits $l = (l_1, \dots, l_K)$, probabilities
$q = \operatorname{softmax}(l)$, so $q_c = e^{l_c}/\sum_k e^{l_k}$. True label $c$; one-hot
$y$ with $y_c = 1$.

**(a) CE with one-hot labels = negative log-likelihood.** The one-hot true distribution is
$p = y$, so

$$
H(y, q) = -\sum_k y_k \log q_k = -\log q_c =: \ell
$$

Over a dataset of $m$ samples:

$$
\mathcal{L} = \frac{1}{m}\sum_{i=1}^m \ell_i = -\frac{1}{m}\log \prod_{i=1}^m q_\theta[c_i \mid x^{(i)}]
$$

The product is the **likelihood** of the data under the model. Hence
$\operatorname{argmin} \mathcal{L} = \operatorname{argmax}$ likelihood — **minimizing
cross-entropy is maximum likelihood estimation**. That is the fundamental justification.

**(b) Stable computation from logits.**

$$
\ell = -\log\frac{e^{l_c}}{\sum_k e^{l_k}} = \log\sum_k e^{l_k} - l_c = \operatorname{logsumexp}(l) - l_c
$$

Numerically, subtract the max: $\operatorname{logsumexp}(l) = l_{\max} + \log\sum_k e^{l_k - l_{\max}}$
— avoids overflow of $e^{l_k}$.

**(c) The gradient is beautifully simple.** Since $\partial\, \operatorname{logsumexp}(l)/\partial l_j = q_j$:

$$
\frac{\partial \ell}{\partial l_j} = q_j - y_j \qquad\text{i.e.}\qquad \nabla_l \ell = \hat{y} - y
$$

Prediction minus truth. No softmax-derivative factors, no vanishing when the model is
confidently wrong.

**(d) Why not MSE for classification?** With softmax + squared error, the gradient w.r.t.
the true class's logit carries a factor $q_c(1 - q_c)$: when the model is confidently
wrong ($q_c \approx 0$), that factor $\approx 0$ and learning stalls exactly when it
matters. Cross-entropy's gradient $q_c - 1 \approx -1$ stays strong. (MSE is fine for
regression; CE is the right pairing with softmax.)

**(e) KL view.** $H(y, q) = H(y) + D_{KL}(y\|q)$ and $H(y) = 0$ for one-hot, so
$\min_q H(y,q) = \min_q D_{KL}(y \| q)$: CE training pulls the predicted distribution
onto the one-hot label as measured by KL.

**(f) Generalizations.** Soft targets $p \ne$ one-hot (label smoothing
$p = (1-\varepsilon)\,\text{onehot}(c) + \varepsilon/K$, knowledge distillation) work because
the full form $-\sum_k p_k \log q_k$ never required one-hot. Temperature scaling
$q_k = \operatorname{softmax}(l_k / T)$ sharpens/flattens the distribution. Via Bayes,
$q(c|x)$ is a posterior and CE = NLL of the posterior up to the prior.

## 10. Worked example (3 classes)

Logits $l = (2, 0, -1)$, true class $c = 0$ so $y = (1, 0, 0)$.

**Softmax:** $e^l = (7.389, 1, 0.368)$, $Z = 8.757$, so

$$
q = \operatorname{softmax}(l) = (0.8438,\; 0.1142,\; 0.0420)
$$

**Loss:** $\ell = \operatorname{logsumexp}(l) - l_0 = \ln 8.757 - 2 = 2.0697 - 2 = 0.0697$
nats, and indeed $-\ln 0.8438 = 0.0697$. ✓

**Gradient:** $\nabla_l \ell = q - y = (-0.1562,\; 0.1142,\; 0.0420)$ — push $l_0$ up,
push the others down, in proportion to how wrong the model is.

If instead the model were confidently wrong, $l = (-2, 0, 1)$ with $c = 0$:
$q_0 = e^{-2}/(e^{-2}+1+e) \approx 0.0420$, $\ell = -\ln 0.0420 \approx 3.17$ nats (large
penalty), and $\partial \ell/\partial l_0 = q_0 - 1 \approx -0.96$ (strong corrective
gradient). Compare MSE+softmax, where the gradient would be scaled by $q_0(1-q_0) \approx 0.04$
— nearly stalled.

## 11. Summary and further reading

- Information must be additive for independent events → uniquely $I(p) = -\log p$ (bits/nats).
- Entropy $H(p)$ = optimal expected code length = compression limit (Kraft + source coding
  theorem); max $\log K$ at uniform; chain rule; conditioning reduces entropy.
- Cross-entropy $H(p,q)$ = expected code length under the wrong model $= H(p) + D_{KL}(p\|q)$.
- KL divergence: non-negative (Gibbs), asymmetric, convex, controls total variation
  (Pinsker); equals mutual information for joint vs. product; equals free-energy difference
  (in $k_B T$ units) relative to a canonical $q$.
- Maximum entropy under constraints → exponential family → Boltzmann in physics,
  **softmax** in classification.
- Classification: one-hot labels make CE $= -\log q[c]$ = NLL, so minimizing CE = MLE =
  minimizing KL to the label; with softmax logits, $\ell = \operatorname{logsumexp}(l) - l_c$
  (numerically stable) and $\nabla_l \ell = \hat y - y$ (never stalls when confidently wrong,
  unlike MSE).

**Further reading:** Shannon (1948) *A Mathematical Theory of Communication*; Cover &
Thomas, *Elements of Information Theory*, ch. 2 & 5; MacKay, *Information Theory, Inference,
and Learning Algorithms*; Kullback & Leibler (1951); Jaynes, *Probability Theory: The Logic
of Science* (max-ent); Bishop, *PRML* §2.4 & §4.3 (KL, logistic regression); Murphy, *Probabilistic
Machine Learning* ch. 2–3.